In [2]:
# # ============================================================
# # IPL CRICKET API INGESTION - BRONZE LAYER
# # Microsoft Fabric Notebook
# # ============================================================

# # ============================================================
# # 1. IMPORTS
# # ============================================================

# import requests
# import json
# import time
# from datetime import datetime, timezone

# from pyspark.sql import Row
# from pyspark.sql.types import (
#     StructType,
#     StructField,
#     StringType,
#     TimestampType,
#     IntegerType
# )


StatementMeta(, 5c20ff4f-1471-4749-8388-e84c396660e3, 4, Finished, Available, Finished, False)

In [3]:
# # ============================================================
# # 2. PIPELINE PARAMETERS
# # ============================================================

# source_name = ""
# endpoint = ""
# target_table = ""
# load_type = ""
# run_id = ""
# rapidapi_key = ""

StatementMeta(, 5c20ff4f-1471-4749-8388-e84c396660e3, 5, Finished, Available, Finished, False)

In [4]:
# # ============================================================
# # 3. PARAMETER VALIDATION
# # ============================================================

# required_parameters = {
#     "source_name": source_name,
#     "endpoint": endpoint,
#     "target_table": target_table,
#     "load_type": load_type,
#     "run_id": run_id
# }

# for parameter_name, parameter_value in required_parameters.items():

#     if parameter_value is None or str(parameter_value).strip() == "":
#         raise ValueError(
#             f"Required parameter '{parameter_name}' is missing."
#         )


# if rapidapi_key is None or str(rapidapi_key).strip() == "":
#     raise ValueError(
#         "RapidAPI key is missing. "
#         "Pass the key securely from the pipeline."
#     )


# print("[VALIDATION] All parameters are valid.")
# print(f"[PARAMETER] Source      : {source_name}")
# print(f"[PARAMETER] Endpoint    : {endpoint}")
# print(f"[PARAMETER] Target      : {target_table}")
# print(f"[PARAMETER] Load Type   : {load_type}")
# print(f"[PARAMETER] Run ID      : {run_id}")

StatementMeta(, 5c20ff4f-1471-4749-8388-e84c396660e3, 6, Finished, Available, Finished, False)

ValueError: Required parameter 'source_name' is missing.

In [11]:
# # ============================================================
# # 4. RAPIDAPI CONFIGURATION
# # ============================================================

# RAPIDAPI_HOST = "cricbuzz-cricket.p.rapidapi.com"

# BASE_URL = f"https://{RAPIDAPI_HOST}"

# api_url = f"{BASE_URL}/{endpoint.lstrip('/')}"


# headers = {
#     "X-RapidAPI-Key": rapidapi_key,
#     "X-RapidAPI-Host": RAPIDAPI_HOST
# }


# print("[API] Host     :", RAPIDAPI_HOST)
# print("[API] Endpoint :", endpoint)
# print("[API] URL      :", api_url)




StatementMeta(, 6130a9ec-71f9-4997-b5d0-00be933f5814, 13, Finished, Available, Finished, False)

[API] Host     : cricbuzz-cricket.p.rapidapi.com
[API] Endpoint : 
[API] URL      : https://cricbuzz-cricket.p.rapidapi.com/


In [12]:
# # ============================================================
# # 5. API REQUEST SETTINGS
# # ============================================================

# MAX_RETRIES = 3
# TIMEOUT_SECONDS = 60

StatementMeta(, 6130a9ec-71f9-4997-b5d0-00be933f5814, 14, Finished, Available, Finished, False)

In [13]:
# # ============================================================
# # 6. API CALL WITH PROPER RETRY LOGIC
# # ============================================================
# #
# # Retry:
# #   - Network errors
# #   - Timeout
# #   - HTTP 429 Rate Limit
# #   - HTTP 500-599 Server errors
# #
# # Do NOT retry:
# #   - HTTP 400
# #   - HTTP 401
# #   - HTTP 403
# #   - Other permanent client errors
# #
# # ============================================================

# response = None
# last_error = None


# for attempt in range(1, MAX_RETRIES + 1):

#     try:

#         print(
#             f"[API] Attempt {attempt}/{MAX_RETRIES} "
#             f"for {source_name}"
#         )

#         response = requests.get(
#             api_url,
#             headers=headers,
#             timeout=TIMEOUT_SECONDS
#         )


#         # ----------------------------------------------------
#         # SUCCESS
#         # ----------------------------------------------------

#         if response.status_code == 200:

#             print("[API] Request successful.")
#             break


#         # ----------------------------------------------------
#         # UNAUTHORIZED
#         # ----------------------------------------------------

#         elif response.status_code == 401:

#             raise ValueError(
#                 "HTTP 401 - Invalid RapidAPI credentials. "
#                 "Check the RapidAPI key and X-RapidAPI-Host."
#             )


#         # ----------------------------------------------------
#         # FORBIDDEN
#         # ----------------------------------------------------

#         elif response.status_code == 403:

#             raise ValueError(
#                 "HTTP 403 - Access forbidden. "
#                 "Check your RapidAPI subscription and endpoint access."
#             )


#         # ----------------------------------------------------
#         # BAD REQUEST
#         # ----------------------------------------------------

#         elif response.status_code == 400:

#             raise ValueError(
#                 f"HTTP 400 - Bad request: "
#                 f"{response.text[:500]}"
#             )


#         # ----------------------------------------------------
#         # RATE LIMIT
#         # ----------------------------------------------------

#         elif response.status_code == 429:

#             if attempt < MAX_RETRIES:

#                 retry_after = response.headers.get(
#                     "Retry-After"
#                 )

#                 if retry_after:

#                     try:
#                         wait_time = int(retry_after)
#                     except:
#                         wait_time = 2 ** attempt

#                 else:

#                     wait_time = 2 ** attempt

#                 print(
#                     f"[API] Rate limited. "
#                     f"Waiting {wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#                 continue

#             else:

#                 raise RuntimeError(
#                     "HTTP 429 - RapidAPI rate limit exceeded."
#                 )


#         # ----------------------------------------------------
#         # SERVER ERROR
#         # ----------------------------------------------------

#         elif 500 <= response.status_code <= 599:

#             if attempt < MAX_RETRIES:

#                 wait_time = 2 ** attempt

#                 print(
#                     f"[API] Server error {response.status_code}. "
#                     f"Retrying in {wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#                 continue

#             else:

#                 raise RuntimeError(
#                     f"HTTP {response.status_code} - "
#                     f"RapidAPI server error."
#                 )


#         # ----------------------------------------------------
#         # OTHER ERROR
#         # ----------------------------------------------------

#         else:

#             raise ValueError(
#                 f"HTTP {response.status_code}: "
#                 f"{response.text[:500]}"
#             )


#     except requests.exceptions.Timeout as e:

#         last_error = e

#         print(
#             f"[API] Timeout on attempt {attempt}."
#         )

#         if attempt < MAX_RETRIES:

#             wait_time = 2 ** attempt

#             print(
#                 f"[API] Retrying in {wait_time} seconds..."
#             )

#             time.sleep(wait_time)

#         else:

#             raise RuntimeError(
#                 "API request timed out after "
#                 f"{MAX_RETRIES} attempts."
#             )


#     except requests.exceptions.RequestException as e:

#         last_error = e

#         print(
#             f"[API] Network error: {str(e)}"
#         )

#         if attempt < MAX_RETRIES:

#             wait_time = 2 ** attempt

#             print(
#                 f"[API] Retrying in {wait_time} seconds..."
#             )

#             time.sleep(wait_time)

#         else:

#             raise RuntimeError(
#                 f"API request failed after "
#                 f"{MAX_RETRIES} attempts: {str(e)}"
#             )


#     except (ValueError, RuntimeError):

#         # Do not retry permanent errors such as 401, 403, 400.

#         raise

StatementMeta(, 6130a9ec-71f9-4997-b5d0-00be933f5814, 15, Finished, Available, Finished, False)

[API] Attempt 1/3 for 


ValueError: HTTP 401 - Invalid RapidAPI credentials. Check the RapidAPI key and X-RapidAPI-Host.

In [ ]:
# # ============================================================
# # 7. VERIFY RESPONSE
# # ============================================================

# if response is None:

#     raise RuntimeError(
#         "No response received from API."
#     )


# if response.status_code != 200:

#     raise RuntimeError(
#         f"API ingestion failed. "
#         f"HTTP Status: {response.status_code}"
#     )


# print("[API] HTTP Status : 200")
# print("[API] API call completed successfully.")


In [ ]:
# # ============================================================
# # 8. PARSE JSON RESPONSE
# # ============================================================

# try:

#     response_json = response.json()

# except ValueError as e:

#     raise RuntimeError(
#         f"API returned invalid JSON: {str(e)}"
#     )


# print("[API] JSON response parsed successfully.")


In [ ]:
# # ============================================================
# # 9. CONVERT RESPONSE TO STRING
# # ============================================================
# #
# # Bronze should preserve the raw API response.
# #
# # We store the complete JSON as a string so that:
# #
# # API
# #   ↓
# # Raw JSON
# #   ↓
# # Bronze
# #
# # Silver will later parse / flatten this JSON.
# #
# # ============================================================

# response_json_string = json.dumps(
#     response_json,
#     ensure_ascii=False
# )


# response_size = len(response_json_string)


# print(
#     f"[API] Response size : {response_size} characters"
# )

In [ ]:
# # ============================================================
# # 10. CREATE BRONZE RECORD
# # ============================================================

# ingested_at = datetime.now(timezone.utc)


# bronze_record = Row(
#     run_id=str(run_id),
#     source_name=str(source_name),
#     endpoint=str(endpoint),
#     load_type=str(load_type),
#     response_json=response_json_string,
#     ingested_at=ingested_at
# )

In [ ]:
# # ============================================================
# # 11. CREATE DATAFRAME
# # ============================================================

# bronze_schema = StructType([

#     StructField(
#         "run_id",
#         StringType(),
#         False
#     ),

#     StructField(
#         "source_name",
#         StringType(),
#         False
#     ),

#     StructField(
#         "endpoint",
#         StringType(),
#         False
#     ),

#     StructField(
#         "load_type",
#         StringType(),
#         True
#     ),

#     StructField(
#         "response_json",
#         StringType(),
#         False
#     ),

#     StructField(
#         "ingested_at",
#         TimestampType(),
#         False
#     )
# ])


# bronze_df = spark.createDataFrame(
#     [bronze_record],
#     schema=bronze_schema
# )


In [ ]:
# # ============================================================
# # 12. DISPLAY DATA BEFORE WRITING
# # ============================================================

# print("[BRONZE] Record prepared.")

# bronze_df.select(
#     "run_id",
#     "source_name",
#     "endpoint",
#     "load_type",
#     "ingested_at"
# ).show(
#     truncate=False
# )



In [ ]:
# # ============================================================
# # 13. CHECK TARGET TABLE
# # ============================================================

# target_table_clean = target_table.strip()

# print(
#     f"[BRONZE] Target table: {target_table_clean}"
# )


In [ ]:
# # ============================================================
# # 14. CREATE TARGET TABLE IF IT DOES NOT EXIST
# # ============================================================

# if not spark.catalog.tableExists(target_table_clean):

#     print(
#         f"[BRONZE] Table {target_table_clean} "
#         "does not exist."
#     )

#     print(
#         "[BRONZE] Creating target table..."
#     )

#     bronze_df.write \
#         .format("delta") \
#         .mode("overwrite") \
#         .saveAsTable(target_table_clean)

# else:

#     print(
#         f"[BRONZE] Table {target_table_clean} "
#         "already exists."
#     )


In [ ]:
# # ============================================================
# # 15. WRITE DATA TO BRONZE
# # ============================================================
# #
# # Append is used so previous Bronze data is preserved.
# #
# # ============================================================

# if spark.catalog.tableExists(target_table_clean):

#     bronze_df.write \
#         .format("delta") \
#         .mode("append") \
#         .saveAsTable(target_table_clean)

In [ ]:
# # ============================================================
# # 16. FINAL VALIDATION
# # ============================================================

# written_count = spark.table(
#     target_table_clean
# ).filter(
#     f"run_id = '{str(run_id)}'"
# ).count()


# print(
#     f"[BRONZE] Records written for this run: "
#     f"{written_count}"
# )


In [ ]:
# # ============================================================
# # 17. FINAL STATUS
# # ============================================================

# print("=" * 60)

# print("IPL API BRONZE INGESTION COMPLETED SUCCESSFULLY")

# print("=" * 60)

# print(f"Source      : {source_name}")
# print(f"Endpoint    : {endpoint}")
# print(f"Target      : {target_table_clean}")
# print(f"Load Type   : {load_type}")
# print(f"Run ID      : {run_id}")
# print(f"Ingested At : {ingested_at}")
# print(f"Rows        : {written_count}")

# print("=" * 60)

In [1]:
# # ============================================================
# # IPL CRICKET API INGESTION - BRONZE LAYER
# # Microsoft Fabric Notebook
# # ============================================================

# # ============================================================
# # 1. IMPORTS
# # ============================================================

# import requests
# import json
# import time
# from datetime import datetime, timezone

# from pyspark.sql import Row
# from pyspark.sql.types import (
#     StructType,
#     StructField,
#     StringType,
#     TimestampType,
#     IntegerType
# )


# # ============================================================
# # 2. PIPELINE PARAMETERS
# # ============================================================
# #
# # These values are passed from Fabric Pipeline:
# #
# # source_name
# # endpoint
# # target_table
# # load_type
# # run_id
# # rapidapi_key
# #
# # IMPORTANT - Cricbuzz Cricket (RapidAPI) endpoints are
# # VERSIONED. Pass the endpoint WITH the "v1" segment, e.g.:
# #
# #   matches/v1/recent     (NOT matches/recent)
# #   matches/v1/upcoming   (NOT matches/upcoming)
# #   matches/v1/live       (NOT matches/live)
# #
# # Check the "Endpoints" tab on the RapidAPI dashboard for the
# # exact path of whichever resource you're ingesting - don't
# # hand-type these from memory in the pipeline parameters.
# #
# # ============================================================

# source_name = ""
# endpoint = ""
# target_table = ""
# load_type = ""
# run_id = ""
# rapidapi_key = ""


# # ============================================================
# # 3. PARAMETER VALIDATION
# # ============================================================

# required_parameters = {
#     "source_name": source_name,
#     "endpoint": endpoint,
#     "target_table": target_table,
#     "load_type": load_type,
#     "run_id": run_id
# }

# for parameter_name, parameter_value in required_parameters.items():

#     if parameter_value is None or str(parameter_value).strip() == "":
#         raise ValueError(
#             f"Required parameter '{parameter_name}' is missing."
#         )


# if rapidapi_key is None or str(rapidapi_key).strip() == "":
#     raise ValueError(
#         "RapidAPI key is missing. "
#         "Pass the key securely from the pipeline."
#     )


# print("[VALIDATION] All parameters are valid.")
# print(f"[PARAMETER] Source      : {source_name}")
# print(f"[PARAMETER] Endpoint    : {endpoint}")
# print(f"[PARAMETER] Target      : {target_table}")
# print(f"[PARAMETER] Load Type   : {load_type}")
# print(f"[PARAMETER] Run ID      : {run_id}")


# # ============================================================
# # 4. RAPIDAPI CONFIGURATION
# # ============================================================

# RAPIDAPI_HOST = "cricbuzz-cricket.p.rapidapi.com"

# BASE_URL = f"https://{RAPIDAPI_HOST}"

# api_url = f"{BASE_URL}/{endpoint.lstrip('/')}"


# headers = {
#     "X-RapidAPI-Key": rapidapi_key,
#     "X-RapidAPI-Host": RAPIDAPI_HOST
# }


# print("[API] Host     :", RAPIDAPI_HOST)
# print("[API] Endpoint :", endpoint)
# print("[API] URL      :", api_url)


# # ============================================================
# # 5. API REQUEST SETTINGS
# # ============================================================

# MAX_RETRIES = 3
# TIMEOUT_SECONDS = 60


# # ============================================================
# # 6. API CALL WITH PROPER RETRY LOGIC
# # ============================================================
# #
# # Retry:
# #   - Network errors
# #   - Timeout
# #   - HTTP 429 Rate Limit
# #   - HTTP 500-599 Server errors
# #
# # Do NOT retry:
# #   - HTTP 400
# #   - HTTP 401
# #   - HTTP 403
# #   - HTTP 404
# #   - Other permanent client errors
# #
# # ============================================================

# response = None
# last_error = None


# for attempt in range(1, MAX_RETRIES + 1):

#     try:

#         print(
#             f"[API] Attempt {attempt}/{MAX_RETRIES} "
#             f"for {source_name}"
#         )

#         response = requests.get(
#             api_url,
#             headers=headers,
#             timeout=TIMEOUT_SECONDS
#         )


#         # ----------------------------------------------------
#         # SUCCESS
#         # ----------------------------------------------------

#         if response.status_code == 200:

#             print("[API] Request successful.")
#             break


#         # ----------------------------------------------------
#         # UNAUTHORIZED
#         # ----------------------------------------------------

#         elif response.status_code == 401:

#             raise ValueError(
#                 f"HTTP 401 - Invalid RapidAPI credentials. "
#                 f"Check the RapidAPI key and X-RapidAPI-Host. "
#                 f"URL: {api_url}"
#             )


#         # ----------------------------------------------------
#         # FORBIDDEN
#         # ----------------------------------------------------

#         elif response.status_code == 403:

#             raise ValueError(
#                 f"HTTP 403 - Access forbidden. "
#                 f"Check your RapidAPI subscription and endpoint access. "
#                 f"URL: {api_url}"
#             )


#         # ----------------------------------------------------
#         # NOT FOUND
#         # ----------------------------------------------------
#         #
#         # Almost always means the 'endpoint' pipeline parameter
#         # is wrong - e.g. missing the versioned path segment
#         # (should be 'matches/v1/upcoming', not 'matches/upcoming').
#         # Verify the exact path on the RapidAPI "Endpoints" tab.
#         #
#         # ----------------------------------------------------

#         elif response.status_code == 404:

#             raise ValueError(
#                 f"HTTP 404 - Endpoint not found. "
#                 f"Check the 'endpoint' pipeline parameter for a "
#                 f"missing/incorrect path segment (e.g. Cricbuzz "
#                 f"endpoints are versioned: 'matches/v1/upcoming', "
#                 f"not 'matches/upcoming'). "
#                 f"URL attempted: {api_url} | "
#                 f"Response: {response.text[:500]}"
#             )


#         # ----------------------------------------------------
#         # BAD REQUEST
#         # ----------------------------------------------------

#         elif response.status_code == 400:

#             raise ValueError(
#                 f"HTTP 400 - Bad request: "
#                 f"{response.text[:500]} | "
#                 f"URL: {api_url}"
#             )


#         # ----------------------------------------------------
#         # RATE LIMIT
#         # ----------------------------------------------------

#         elif response.status_code == 429:

#             if attempt < MAX_RETRIES:

#                 retry_after = response.headers.get(
#                     "Retry-After"
#                 )

#                 if retry_after:

#                     try:
#                         wait_time = int(retry_after)
#                     except:
#                         wait_time = 2 ** attempt

#                 else:

#                     wait_time = 2 ** attempt

#                 print(
#                     f"[API] Rate limited. "
#                     f"Waiting {wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#                 continue

#             else:

#                 raise RuntimeError(
#                     f"HTTP 429 - RapidAPI rate limit exceeded. "
#                     f"URL: {api_url}"
#                 )


#         # ----------------------------------------------------
#         # SERVER ERROR
#         # ----------------------------------------------------

#         elif 500 <= response.status_code <= 599:

#             if attempt < MAX_RETRIES:

#                 wait_time = 2 ** attempt

#                 print(
#                     f"[API] Server error {response.status_code}. "
#                     f"Retrying in {wait_time} seconds..."
#                 )

#                 time.sleep(wait_time)

#                 continue

#             else:

#                 raise RuntimeError(
#                     f"HTTP {response.status_code} - "
#                     f"RapidAPI server error. "
#                     f"URL: {api_url}"
#                 )


#         # ----------------------------------------------------
#         # OTHER ERROR
#         # ----------------------------------------------------

#         else:

#             raise ValueError(
#                 f"HTTP {response.status_code}: "
#                 f"{response.text[:500]} | "
#                 f"URL: {api_url}"
#             )


#     except requests.exceptions.Timeout as e:

#         last_error = e

#         print(
#             f"[API] Timeout on attempt {attempt}."
#         )

#         if attempt < MAX_RETRIES:

#             wait_time = 2 ** attempt

#             print(
#                 f"[API] Retrying in {wait_time} seconds..."
#             )

#             time.sleep(wait_time)

#         else:

#             raise RuntimeError(
#                 f"API request timed out after "
#                 f"{MAX_RETRIES} attempts. URL: {api_url}"
#             )


#     except requests.exceptions.RequestException as e:

#         last_error = e

#         print(
#             f"[API] Network error: {str(e)}"
#         )

#         if attempt < MAX_RETRIES:

#             wait_time = 2 ** attempt

#             print(
#                 f"[API] Retrying in {wait_time} seconds..."
#             )

#             time.sleep(wait_time)

#         else:

#             raise RuntimeError(
#                 f"API request failed after "
#                 f"{MAX_RETRIES} attempts: {str(e)}. URL: {api_url}"
#             )


#     except (ValueError, RuntimeError):

#         # Do not retry permanent errors such as 400, 401, 403, 404.

#         raise


# # ============================================================
# # 7. VERIFY RESPONSE
# # ============================================================

# if response is None:

#     raise RuntimeError(
#         f"No response received from API. URL: {api_url}"
#     )


# if response.status_code != 200:

#     raise RuntimeError(
#         f"API ingestion failed. "
#         f"HTTP Status: {response.status_code}. URL: {api_url}"
#     )


# print("[API] HTTP Status : 200")
# print("[API] API call completed successfully.")


# # ============================================================
# # 8. PARSE JSON RESPONSE
# # ============================================================

# try:

#     response_json = response.json()

# except ValueError as e:

#     raise RuntimeError(
#         f"API returned invalid JSON: {str(e)}"
#     )


# print("[API] JSON response parsed successfully.")


# # ============================================================
# # 9. CONVERT RESPONSE TO STRING
# # ============================================================
# #
# # Bronze should preserve the raw API response.
# #
# # We store the complete JSON as a string so that:
# #
# # API
# #   ↓
# # Raw JSON
# #   ↓
# # Bronze
# #
# # Silver will later parse / flatten this JSON.
# #
# # ============================================================

# response_json_string = json.dumps(
#     response_json,
#     ensure_ascii=False
# )


# response_size = len(response_json_string)


# print(
#     f"[API] Response size : {response_size} characters"
# )


# # ============================================================
# # 10. CREATE BRONZE RECORD
# # ============================================================

# ingested_at = datetime.now(timezone.utc)


# bronze_record = Row(
#     run_id=str(run_id),
#     source_name=str(source_name),
#     endpoint=str(endpoint),
#     load_type=str(load_type),
#     response_json=response_json_string,
#     ingested_at=ingested_at
# )


# # ============================================================
# # 11. CREATE DATAFRAME
# # ============================================================

# bronze_schema = StructType([

#     StructField(
#         "run_id",
#         StringType(),
#         False
#     ),

#     StructField(
#         "source_name",
#         StringType(),
#         False
#     ),

#     StructField(
#         "endpoint",
#         StringType(),
#         False
#     ),

#     StructField(
#         "load_type",
#         StringType(),
#         True
#     ),

#     StructField(
#         "response_json",
#         StringType(),
#         False
#     ),

#     StructField(
#         "ingested_at",
#         TimestampType(),
#         False
#     )
# ])


# bronze_df = spark.createDataFrame(
#     [bronze_record],
#     schema=bronze_schema
# )


# # ============================================================
# # 12. DISPLAY DATA BEFORE WRITING
# # ============================================================

# print("[BRONZE] Record prepared.")

# bronze_df.select(
#     "run_id",
#     "source_name",
#     "endpoint",
#     "load_type",
#     "ingested_at"
# ).show(
#     truncate=False
# )


# # ============================================================
# # 13. CHECK TARGET TABLE
# # ============================================================

# target_table_clean = target_table.strip()

# print(
#     f"[BRONZE] Target table: {target_table_clean}"
# )


# # ============================================================
# # 14. CREATE TARGET TABLE IF IT DOES NOT EXIST
# # ============================================================

# if not spark.catalog.tableExists(target_table_clean):

#     print(
#         f"[BRONZE] Table {target_table_clean} "
#         "does not exist."
#     )

#     print(
#         "[BRONZE] Creating target table..."
#     )

#     bronze_df.write \
#         .format("delta") \
#         .mode("overwrite") \
#         .saveAsTable(target_table_clean)

# else:

#     print(
#         f"[BRONZE] Table {target_table_clean} "
#         "already exists."
#     )


# # ============================================================
# # 15. WRITE DATA TO BRONZE
# # ============================================================
# #
# # Append is used so previous Bronze data is preserved.
# #
# # ============================================================

# if spark.catalog.tableExists(target_table_clean):

#     bronze_df.write \
#         .format("delta") \
#         .mode("append") \
#         .saveAsTable(target_table_clean)


# # ============================================================
# # 16. FINAL VALIDATION
# # ============================================================

# written_count = spark.table(
#     target_table_clean
# ).filter(
#     f"run_id = '{str(run_id)}'"
# ).count()


# print(
#     f"[BRONZE] Records written for this run: "
#     f"{written_count}"
# )


# # ============================================================
# # 17. FINAL STATUS
# # ============================================================

# print("=" * 60)

# print("IPL API BRONZE INGESTION COMPLETED SUCCESSFULLY")

# print("=" * 60)

# print(f"Source      : {source_name}")
# print(f"Endpoint    : {endpoint}")
# print(f"Target      : {target_table_clean}")
# print(f"Load Type   : {load_type}")
# print(f"Run ID      : {run_id}")
# print(f"Ingested At : {ingested_at}")
# print(f"Rows        : {written_count}")

# print("=" * 60)

StatementMeta(, 02a010ce-83c8-4b8a-9f85-f24fa6d4bfce, 3, Finished, Available, Finished, False)

ValueError: Required parameter 'source_name' is missing.

In [ ]:
# ============================================================
# IPL CRICKET API INGESTION - BRONZE LAYER
# Microsoft Fabric Notebook
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================

import requests
import json
import time
from datetime import datetime, timezone

from pyspark.sql import Row
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    TimestampType,
    IntegerType
)

In [ ]:
# ============================================================
# 2. PIPELINE PARAMETERS
# ============================================================
#
# These values are passed from Fabric Pipeline:
#
# source_name
# endpoint
# target_table
# load_type
# run_id
# rapidapi_key
#
# IMPORTANT - Cricbuzz Cricket (RapidAPI) endpoints are
# VERSIONED. Pass the endpoint WITH the "v1" segment, e.g.:
#
#   matches/v1/recent     (NOT matches/recent)
#   matches/v1/upcoming   (NOT matches/upcoming)
#   matches/v1/live       (NOT matches/live)
#
# Check the "Endpoints" tab on the RapidAPI dashboard for the
# exact path of whichever resource you're ingesting - don't
# hand-type these from memory in the pipeline parameters.
#
# ============================================================

source_name = ""
endpoint = ""
target_table = ""
load_type = ""
run_id = ""
rapidapi_key = ""



In [ ]:
# ============================================================
# 3. PARAMETER VALIDATION
# ============================================================

required_parameters = {
    "source_name": source_name,
    "endpoint": endpoint,
    "target_table": target_table,
    "load_type": load_type,
    "run_id": run_id
}

for parameter_name, parameter_value in required_parameters.items():

    if parameter_value is None or str(parameter_value).strip() == "":
        raise ValueError(
            f"Required parameter '{parameter_name}' is missing."
        )


if rapidapi_key is None or str(rapidapi_key).strip() == "":
    raise ValueError(
        "RapidAPI key is missing. "
        "Pass the key securely from the pipeline."
    )


print("[VALIDATION] All parameters are valid.")
print(f"[PARAMETER] Source      : {source_name}")
print(f"[PARAMETER] Endpoint    : {endpoint}")
print(f"[PARAMETER] Target      : {target_table}")
print(f"[PARAMETER] Load Type   : {load_type}")
print(f"[PARAMETER] Run ID      : {run_id}")



In [ ]:
# ============================================================
# 4. RAPIDAPI CONFIGURATION
# ============================================================

RAPIDAPI_HOST = "cricbuzz-cricket.p.rapidapi.com"

BASE_URL = f"https://{RAPIDAPI_HOST}"

api_url = f"{BASE_URL}/{endpoint.lstrip('/')}"


headers = {
    "X-RapidAPI-Key": rapidapi_key,
    "X-RapidAPI-Host": RAPIDAPI_HOST
}


print("[API] Host     :", RAPIDAPI_HOST)
print("[API] Endpoint :", endpoint)
print("[API] URL      :", api_url)


In [ ]:
# ============================================================
# 5. API REQUEST SETTINGS
# ============================================================

MAX_RETRIES = 3
TIMEOUT_SECONDS = 60

In [ ]:
# ============================================================
# 6. API CALL WITH PROPER RETRY LOGIC
# ============================================================
#
# Retry:
#   - Network errors
#   - Timeout
#   - HTTP 429 Rate Limit
#   - HTTP 500-599 Server errors
#
# Do NOT retry:
#   - HTTP 400
#   - HTTP 401
#   - HTTP 403
#   - HTTP 404
#   - Other permanent client errors
#
# ============================================================

response = None
last_error = None


for attempt in range(1, MAX_RETRIES + 1):

    try:

        print(
            f"[API] Attempt {attempt}/{MAX_RETRIES} "
            f"for {source_name}"
        )

        response = requests.get(
            api_url,
            headers=headers,
            timeout=TIMEOUT_SECONDS
        )


        # ----------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------

        if response.status_code == 200:

            print("[API] Request successful.")
            break


        # ----------------------------------------------------
        # UNAUTHORIZED
        # ----------------------------------------------------

        elif response.status_code == 401:

            raise ValueError(
                f"HTTP 401 - Invalid RapidAPI credentials. "
                f"Check the RapidAPI key and X-RapidAPI-Host. "
                f"URL: {api_url}"
            )


        # ----------------------------------------------------
        # FORBIDDEN
        # ----------------------------------------------------

        elif response.status_code == 403:

            raise ValueError(
                f"HTTP 403 - Access forbidden. "
                f"Check your RapidAPI subscription and endpoint access. "
                f"URL: {api_url}"
            )


        # ----------------------------------------------------
        # NOT FOUND
        # ----------------------------------------------------
        #
        # Almost always means the 'endpoint' pipeline parameter
        # is wrong - e.g. missing the versioned path segment
        # (should be 'matches/v1/upcoming', not 'matches/upcoming').
        # Verify the exact path on the RapidAPI "Endpoints" tab.
        #
        # ----------------------------------------------------

        elif response.status_code == 404:

            raise ValueError(
                f"HTTP 404 - Endpoint not found. "
                f"Check the 'endpoint' pipeline parameter for a "
                f"missing/incorrect path segment (e.g. Cricbuzz "
                f"endpoints are versioned: 'matches/v1/upcoming', "
                f"not 'matches/upcoming'). "
                f"URL attempted: {api_url} | "
                f"Response: {response.text[:500]}"
            )


        # ----------------------------------------------------
        # BAD REQUEST
        # ----------------------------------------------------

        elif response.status_code == 400:

            raise ValueError(
                f"HTTP 400 - Bad request: "
                f"{response.text[:500]} | "
                f"URL: {api_url}"
            )


        # ----------------------------------------------------
        # RATE LIMIT
        # ----------------------------------------------------

        elif response.status_code == 429:

            if attempt < MAX_RETRIES:

                retry_after = response.headers.get(
                    "Retry-After"
                )

                if retry_after:

                    try:
                        wait_time = int(retry_after)
                    except:
                        wait_time = 2 ** attempt

                else:

                    wait_time = 2 ** attempt

                print(
                    f"[API] Rate limited. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)

                continue

            else:

                raise RuntimeError(
                    f"HTTP 429 - RapidAPI rate limit exceeded. "
                    f"URL: {api_url}"
                )


        # ----------------------------------------------------
        # SERVER ERROR
        # ----------------------------------------------------

        elif 500 <= response.status_code <= 599:

            if attempt < MAX_RETRIES:

                wait_time = 2 ** attempt

                print(
                    f"[API] Server error {response.status_code}. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

                continue

            else:

                raise RuntimeError(
                    f"HTTP {response.status_code} - "
                    f"RapidAPI server error. "
                    f"URL: {api_url}"
                )


        # ----------------------------------------------------
        # OTHER ERROR
        # ----------------------------------------------------

        else:

            raise ValueError(
                f"HTTP {response.status_code}: "
                f"{response.text[:500]} | "
                f"URL: {api_url}"
            )


    except requests.exceptions.Timeout as e:

        last_error = e

        print(
            f"[API] Timeout on attempt {attempt}."
        )

        if attempt < MAX_RETRIES:

            wait_time = 2 ** attempt

            print(
                f"[API] Retrying in {wait_time} seconds..."
            )

            time.sleep(wait_time)

        else:

            raise RuntimeError(
                f"API request timed out after "
                f"{MAX_RETRIES} attempts. URL: {api_url}"
            )


    except requests.exceptions.RequestException as e:

        last_error = e

        print(
            f"[API] Network error: {str(e)}"
        )

        if attempt < MAX_RETRIES:

            wait_time = 2 ** attempt

            print(
                f"[API] Retrying in {wait_time} seconds..."
            )

            time.sleep(wait_time)

        else:

            raise RuntimeError(
                f"API request failed after "
                f"{MAX_RETRIES} attempts: {str(e)}. URL: {api_url}"
            )


    except (ValueError, RuntimeError):

        # Do not retry permanent errors such as 400, 401, 403, 404.

        raise

In [ ]:
# ============================================================
# 7. VERIFY RESPONSE
# ============================================================

if response is None:

    raise RuntimeError(
        f"No response received from API. URL: {api_url}"
    )


if response.status_code != 200:

    raise RuntimeError(
        f"API ingestion failed. "
        f"HTTP Status: {response.status_code}. URL: {api_url}"
    )


print("[API] HTTP Status : 200")
print("[API] API call completed successfully.")

In [ ]:
# ============================================================
# 8. PARSE JSON RESPONSE
# ============================================================

try:

    response_json = response.json()

except ValueError as e:

    raise RuntimeError(
        f"API returned invalid JSON: {str(e)}"
    )


print("[API] JSON response parsed successfully.")


In [ ]:
# ============================================================
# 9. CONVERT RESPONSE TO STRING
# ============================================================
#
# Bronze should preserve the raw API response.
#
# We store the complete JSON as a string so that:
#
# API
#   ↓
# Raw JSON
#   ↓
# Bronze
#
# Silver will later parse / flatten this JSON.
#
# ============================================================

response_json_string = json.dumps(
    response_json,
    ensure_ascii=False
)


response_size = len(response_json_string)


print(
    f"[API] Response size : {response_size} characters"
)





In [ ]:
# ============================================================
# 10. CREATE BRONZE RECORD
# ============================================================

ingested_at = datetime.now(timezone.utc)


bronze_record = Row(
    run_id=str(run_id),
    source_name=str(source_name),
    endpoint=str(endpoint),
    load_type=str(load_type),
    response_json=response_json_string,
    ingested_at=ingested_at
)


In [ ]:
# ============================================================
# 11. CREATE DATAFRAME
# ============================================================

bronze_schema = StructType([

    StructField(
        "run_id",
        StringType(),
        False
    ),

    StructField(
        "source_name",
        StringType(),
        False
    ),

    StructField(
        "endpoint",
        StringType(),
        False
    ),

    StructField(
        "load_type",
        StringType(),
        True
    ),

    StructField(
        "response_json",
        StringType(),
        False
    ),

    StructField(
        "ingested_at",
        TimestampType(),
        False
    )
])


bronze_df = spark.createDataFrame(
    [bronze_record],
    schema=bronze_schema
)



In [ ]:
# ============================================================
# 12. DISPLAY DATA BEFORE WRITING
# ============================================================

print("[BRONZE] Record prepared.")

bronze_df.select(
    "run_id",
    "source_name",
    "endpoint",
    "load_type",
    "ingested_at"
).show(
    truncate=False
)


In [ ]:
# ============================================================
# 13. CHECK TARGET TABLE
# ============================================================

target_table_clean = target_table.strip()

print(
    f"[BRONZE] Target table: {target_table_clean}"
)


In [ ]:
# ============================================================
# 13. CHECK TARGET TABLE
# ============================================================

target_table_clean = target_table.strip()

print(f"[BRONZE] Target table: {target_table_clean}")


# ============================================================
# 14. CREATE TARGET TABLE IF IT DOES NOT EXIST (schema only)
# ============================================================

if not spark.catalog.tableExists(target_table_clean):

    print(f"[BRONZE] Table {target_table_clean} does not exist. Creating empty table...")

    empty_df = spark.createDataFrame([], schema=bronze_schema)

    empty_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table_clean)

else:

    print(f"[BRONZE] Table {target_table_clean} already exists.")


# ============================================================
# 15. WRITE DATA TO BRONZE (always append — exactly once)
# ============================================================

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(target_table_clean)

print(f"[BRONZE] Row appended to {target_table_clean}.")

In [ ]:

# ============================================================
# 16. FINAL VALIDATION
# ============================================================

written_count = spark.table(
    target_table_clean
).filter(
    f"run_id = '{str(run_id)}'"
).count()


print(
    f"[BRONZE] Records written for this run: "
    f"{written_count}"
)



In [ ]:
# ============================================================
# 17. FINAL STATUS
# ============================================================

print("=" * 60)

print("IPL API BRONZE INGESTION COMPLETED SUCCESSFULLY")

print("=" * 60)

print(f"Source      : {source_name}")
print(f"Endpoint    : {endpoint}")
print(f"Target      : {target_table_clean}")
print(f"Load Type   : {load_type}")
print(f"Run ID      : {run_id}")
print(f"Ingested At : {ingested_at}")
print(f"Rows        : {written_count}")

print("=" * 60)